In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

# Actor: 상태 -> 행동 확률
actor = nn.Sequential(
    nn.Linear(1, 8),
    nn.Tanh(),
    nn.Linear(8, 2),
)

# Critic: 상태 + 행동 -> Q 추정
critic = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 1),
)

state = torch.tensor([1.0])

# Actor
dist = Categorical(logits=actor(state))
action = dist.sample()
log_prob = dist.log_prob(action)

# Score는 Actor를 미분해서 얻는다
score = torch.cat([
    grad.flatten()
    for grad in torch.autograd.grad(
        log_prob, actor.parameters(), retain_graph=True
    )
])

# Critic
critic_input = torch.cat([state, action.float().reshape(1)])
q_hat = critic(critic_input)

# 비용 최소화 관점의 Actor 손실
actor_loss = log_prob * q_hat.detach().squeeze()

In [4]:
for n, p in actor.named_parameters():
  print(n, p.shape)

0.weight torch.Size([8, 1])
0.bias torch.Size([8])
2.weight torch.Size([2, 8])
2.bias torch.Size([2])
